# Lesson 13 – Rewards and Group-Relative Advantages

## Chapter 6 Connection

Chapter 6 moves from **inference-time scaling** to **training-time scaling**.

Previous lessons improved output by spending more compute during generation:

- generate several candidates and select the best
- refine one answer repeatedly

Chapter 6 introduces reinforcement learning, where reward signals are used to update model behavior during training.

For Weird AI, this lesson focuses on the first major step: **turning evaluations into rewards**.


## 1. Inference-Time Scaling vs. Training-Time Scaling

Inference-time scaling improves a single answer by spending more compute during generation.

Training-time scaling improves future answers by updating model weights.

```text
Inference-time scaling:
Prompt → More generation work → Better current answer

Training-time scaling:
Prompt → Generate outputs → Score outputs → Update model → Better future answers
```


In [ ]:
inference_time_examples = [
    "chain-of-thought prompting",
    "best-of-N generation",
    "self-refinement",
]

training_time_examples = [
    "supervised fine-tuning",
    "RLHF",
    "RLVR",
    "GRPO",
]

print("Inference-time scaling:")
for item in inference_time_examples:
    print("-", item)

print("\nTraining-time scaling:")
for item in training_time_examples:
    print("-", item)


## 2. RLHF vs. RLVR

Chapter 6 contrasts two reinforcement learning approaches.

### RLHF

Reinforcement Learning with Human Feedback uses human preference data.

```text
Prompt
  ↓
Multiple responses
  ↓
Humans rank responses
  ↓
Train reward model
  ↓
Use reward model for RL
```

### RLVR

Reinforcement Learning with Verifiable Rewards uses deterministic verification.

```text
Math problem
  ↓
Generated answer
  ↓
Verifier checks correctness
  ↓
Reward = 1 or 0
```

Weird AI is somewhere in between. We do not have a perfect verifier, but we do have a project-specific evaluator.


## 3. What Is a Reward?

A reward is a number that tells a training algorithm how desirable an output was.

For math:

```text
Correct answer → reward 1
Incorrect answer → reward 0
```

For Weird AI:

```text
Good rhyme + good structure + consistent syllables → higher reward
Weak rhyme + poor structure + inconsistent syllables → lower reward
```

The reward is not perfect. It is a signal.


In [ ]:
evaluations = [
    {"text": "Candidate A", "overall_score": 0.20},
    {"text": "Candidate B", "overall_score": 0.65},
    {"text": "Candidate C", "overall_score": 0.90},
]

for item in evaluations:
    reward = item["overall_score"]
    print(item["text"], "reward =", reward)


## 4. Weird AI Rewards Are Heuristic

The book's math verifier is deterministic. If the final answer matches the reference answer, the reward is clear.

Creative writing is different.

A parody can be:

- funny but structurally messy
- well-structured but boring
- rhyming but repetitive
- emotionally accurate but not funny

Weird AI's reward system is therefore **heuristic**, not perfectly verifiable.


### Discussion

What could go wrong if Weird AI only rewards rhyme score?


In [ ]:
candidate = '''
My code cried tonight
The bugs all took flight
My loops felt right
In pale moonlight
'''

fake_scores = {
    "rhyme_score": 0.95,
    "syllable_consistency_score": 0.80,
    "structure_score": 0.90,
    "originality_score": 0.30,
}

print(candidate)
print(fake_scores)


## 5. Rollouts

In reinforcement learning for LLMs, a rollout is a generated response.

For one prompt, we generate several rollouts.

```text
Prompt: Write a parody about SQL joins.

Rollout 1: ...
Rollout 2: ...
Rollout 3: ...
Rollout 4: ...
```

Each rollout receives a reward.


In [ ]:
prompt = "Write a parody about SQL joins."

rollouts = [
    "My left join left me all alone!",
    "Inner joins inside my heart!!",
    "A query with no where clause!!!!",
    "Tables drifting in the night!",
]

for i, rollout in enumerate(rollouts, start=1):
    print(f"Rollout {i}: {rollout}")


## 6. Group-Relative Advantages

GRPO compares rollouts within a group.

Instead of only asking:

> Was this rollout good?

We ask:

> Was this rollout better or worse than the other rollouts for the same prompt?

That comparison is called an **advantage**.

```text
advantage = reward - average group reward
```


In [ ]:
rewards = [0.25, 0.50, 1.00, 0.25]

average_reward = sum(rewards) / len(rewards)
advantages = [reward - average_reward for reward in rewards]

print("Rewards:", rewards)
print("Average reward:", average_reward)
print("Advantages:", advantages)

for i, (reward, advantage) in enumerate(zip(rewards, advantages), start=1):
    print(f"Rollout {i}: reward={reward:.2f}, advantage={advantage:+.2f}")


## 7. Interpreting Advantages

A positive advantage means:

> This rollout scored above the group average.

A negative advantage means:

> This rollout scored below the group average.

In a future training loop, positive-advantage outputs would be reinforced, while negative-advantage outputs would be discouraged.


In [ ]:
for i, advantage in enumerate(advantages, start=1):
    if advantage > 0:
        interpretation = "reinforce this kind of output"
    elif advantage < 0:
        interpretation = "discourage this kind of output"
    else:
        interpretation = "average output"

    print(f"Rollout {i}: {interpretation}")


## 8. Reward Hacking

Reward hacking happens when a model learns to maximize the reward without actually improving the thing humans care about.

For Weird AI, possible reward hacking examples:

- repeating easy rhyming words
- using very short lines to improve syllable consistency
- writing nonsense that rhymes
- copying the same structure every time

A reward function should guide learning, but it should not replace human judgment.


In [ ]:
reward_hacked_lyrics = '''
night
fight
light
right
'''

print(reward_hacked_lyrics)
print("This might rhyme well, but is it a good parody?")


## 9. Mapping Chapter 6 to Weird AI

| Chapter 6 term | Weird AI version |
|---|---|
| Prompt | Parody request |
| Rollout | Generated lyric candidate |
| Verifier/reward model | Weird AI evaluator |
| Reward | Normalized overall score |
| Group | Several lyrics from the same prompt |
| Advantage | Reward minus group average |
| GRPO training data | Prompt, rollout, reward, advantage records |

This lesson prepares those records. It does not update model weights yet.


## 10. Design Activity

Design a reward function for Weird AI.

Answer these questions:

1. Which score should matter most?
2. Should rhyme and syllable scores be weighted equally?
3. Should very short outputs be penalized?
4. Should repeated words be penalized?
5. What kind of reward hacking are you most worried about?


In [ ]:
# Write your reward design notes here.

reward_design = {
    "most_important_score": "TODO",
    "weights": {
        "rhyme": "TODO",
        "syllables": "TODO",
        "structure": "TODO",
    },
    "penalties": ["TODO"],
    "reward_hacking_risks": ["TODO"],
}

reward_design


## 11. Reflection

1. What is the difference between inference-time scaling and training-time scaling?
2. How is RLHF different from RLVR?
3. Why are math rewards easier to verify than parody rewards?
4. What does a positive advantage mean?
5. What does a negative advantage mean?
6. Why might a model exploit a poorly designed reward function?
7. Why is this lesson useful even though we are not training model weights yet?
